In [2]:
from __future__ import annotations

import errno

import lmdb
import time
from pathlib import Path
from typing import Iterable, Iterator, Optional
import logging
import os
import re
from importlib import reload

# from scripts.regsetup import description

import record_pb2
from meta import Run, Dir, File

In [3]:
from db import Db

db_name = 'test-db'
path = Path(db_name)

readonly = True
readonly = False

create=False
# create=True

db = Db.open(path, readonly=readonly, create=create)
env = db.env

In [4]:
env, env.flags(), env.path(), env.info()

(<Environment at 0x225199c1e90>,
 {'subdir': True,
  'readonly': False,
  'metasync': True,
  'sync': True,
  'map_async': False,
  'readahead': True,
  'writemap': False,
  'meminit': False,
  'lock': True},
 'test-db',
 {'map_addr': 0,
  'map_size': 4294967296,
  'last_pgno': 7,
  'last_txnid': 14,
  'max_readers': 126,
  'num_readers': 1})

In [11]:
with env.begin(
    write=False,
    # buffers= ?
) as txn:
    with txn.cursor() as cur:
        for key, value in cur.iternext():
            print(key, value)

b'0042' b'test write Run rec'
b'0049' b'test write Run rec'
b'r:0042' b'test write Run rec'
b'r:0047' b'test write Run rec'
b'r:0048' b'test write Run rec'
b'r:0049' b'test write Run rec'
b'r:0050' b'test write Run rec'
b'r:0051' b'\x08\x01\x1a run from jupyter for development"\x03bla-\xb6\xf3\x9d?5\xb6\xf3\x9d?:\x0binitialized@\x08H\x18'
b'r:0052' b'\x08\x01\x1a run from jupyter for development"\x03bla-\xb6\xf3\x9d?5\xb6\xf3\x9d?:\x0binitialized@\x08H\x18R\x0b\n\x04fld1\x12\x03blaR\x16\n\x04fld2\x12\x0esan 64\r\nder 19'
b'r:0053' b'\x08\x01\x1a run from jupyter for development"\x03bla-\xb6\xf3\x9d?5\xb6\xf3\x9d?:\x0binitialized@\x08H\x18R\x0b\n\x04fld1\x12\x03blaR\x16\n\x04fld2\x12\x0esan 64\r\nder 19'


In [156]:
env.close()

In [9]:
run_id = '0053'
run = Run(
    run_id,
    path='',
    description='run from jupyter for development',
    specification='bla',
    start_time=1.234,
    end_time=1.234,
    duration=1.234,
    num_dirs=8,
    num_files=24,
    extra={'fld1': 'bla', 'fld2': 'san 64\r\nder 19'},
    status='initialized')

from db import Db
key = Db.mk_run_key(run_id)
value = Db.mk_run_rec(run)
# value = msg.SerializeToString()

key, value


(b'r:0053',
 b'\x08\x01\x1a run from jupyter for development"\x03bla-\xb6\xf3\x9d?5\xb6\xf3\x9d?:\x0binitialized@\x08H\x18R\x0b\n\x04fld1\x12\x03blaR\x16\n\x04fld2\x12\x0esan 64\r\nder 19')

In [10]:
with env.begin(write=True) as txn:
    txn.put(key, value)

txn

In [153]:
txn
# txn.stat()